# Leave-One-User-Out Cross-Validation Data Audit

This notebook contains the data validation and outlier analysis performed to investigate why **User 1** (`user1`) performs substantially worse (F1 ≈ 0.85, WER ≈ 0.25) compared to the other three users (F1 ≈ 0.92, WER ≈ 0.02) under Leave-One-User-Out (LOUO) cross-validation.

We analyze the data across four dimensions:
1. **Hand Size Analysis**: Checking if User 1 has significantly different hand dimensions.
2. **Palm Orientation Analysis**: Examining palm Roll, Pitch, and Yaw distributions.
3. **Feature Distribution Analysis**: Reviewing the normalized features fed directly into the Transformer model.
4. **Missing Frame Analysis**: Assessing tracking stability and dropouts across sequences.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set up workspace paths and mock torch to load the pipeline config
class DummyDevice:
    def __init__(self, *args, **kwargs): pass
    def __str__(self): return "cpu"
    def __repr__(self): return "device(type='cpu')"

class DummyTensor:
    pass

class DummyTorch:
    Tensor = DummyTensor
    device = DummyDevice
    @staticmethod
    def manual_seed(seed): pass
    class cuda:
        @staticmethod
        def is_available(): return False
        @staticmethod
        def manual_seed_all(seed): pass
    class backends:
        class cudnn:
            deterministic = True
            benchmark = False

sys.modules['torch'] = DummyTorch
sys.path.append(str(Path.cwd().parent / "tchct_net"))

from config import FEATURE_KEYS, FEATURE_INDEX, PALM_TRIPLETS, HANDS
from features import palm_reference_normalize_sequence

# Setup style
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "font.family": "sans-serif",
    "figure.titlesize": 20,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.figsize": (10, 6)
})

DATASET_ROOT = Path("c:/Shoab/Thesis/Experiments/dataset")
USERS = ["user1", "user2", "user3", "user5"]
USER_MAP = {
    "user1": "User1 (user1)",
    "user2": "User2 (user2)",
    "user3": "User3 (user3)",
    "user5": "User4 (user5)"
}
COLORS = ["#d62728", "#1f77b4", "#2ca02c", "#ff7f0e"]
USER_COLORS = {user: COLORS[i] for i, user in enumerate(USERS)}

## Data Extraction Pipeline
We run a single pass over all CSV files to extract metrics for all analyses.

In [ ]:
def get_dist(df, p1_cols, p2_cols):
    dx = df[p1_cols[0]] - df[p2_cols[0]]
    dy = df[p1_cols[1]] - df[p2_cols[1]]
    dz = df[p1_cols[2]] - df[p2_cols[2]]
    return np.sqrt(dx**2 + dy**2 + dz**2)

def longest_consecutive_missing(missing_mask):
    max_consec = 0
    curr_consec = 0
    for m in missing_mask:
        if m:
            curr_consec += 1
            max_consec = max(max_consec, curr_consec)
        else:
            curr_consec = 0
    return max_consec

hand_size_records = []
orientation_records = []
missing_frame_records = []
feature_dist_samples = {u: [] for u in USERS}

representative_joints = {
    "Palm": ("palm_x", "palm_y", "palm_z"),
    "Wrist": ("wrist_x", "wrist_y", "wrist_z"),
    "Thumb Tip": ("thumb_distal_sx", "thumb_distal_sy", "thumb_distal_sz"),
    "Index Tip": ("index_distal_sx", "index_distal_sy", "index_distal_sz"),
    "Middle Tip": ("middle_distal_sx", "middle_distal_sy", "middle_distal_sz"),
    "Pinky Tip": ("pinky_distal_sx", "pinky_distal_sy", "pinky_distal_sz")
}

for user in USERS:
    user_dir = DATASET_ROOT / user / "leap_data"
    csv_files = sorted(list(user_dir.glob("*.csv")))
    
    for f_path in tqdm(csv_files, desc=f"Processing {user}"):
        seq_name = f_path.stem
        df = pd.read_csv(f_path)
        if len(df) == 0:
            continue
            
        # 1. Missing Frame Analysis
        left_visible = (df["left_confidence"] > 0) & (df["left_palm_x"] != 0)
        right_visible = (df["right_confidence"] > 0) & (df["right_palm_x"] != 0)
        missing_mask = ~(left_visible | right_visible)
        
        num_missing = missing_mask.sum()
        total_len = len(df)
        pct_missing = (num_missing / total_len) * 100.0
        longest_consec = longest_consecutive_missing(missing_mask)
        
        missing_frame_records.append({
            "User": USER_MAP[user],
            "Sequence": seq_name,
            "TotalFrames": total_len,
            "MissingFrames": num_missing,
            "PctMissing": pct_missing,
            "LongestConsecutiveMissing": longest_consec
        })
        
        # 2. Hand Size & Orientation Analysis
        l_hand_sizes_wrist_mid, l_hand_sizes_palm_width, l_hand_sizes_mid_bone = [], [], []
        r_hand_sizes_wrist_mid, r_hand_sizes_palm_width, r_hand_sizes_mid_bone = [], [], []
        l_orientations, r_orientations = [], []
        
        # Left Hand
        l_val_idx = df[left_visible].index
        if len(l_val_idx) > 0:
            l_wrist_mid = get_dist(df.loc[l_val_idx], 
                                   ["left_wrist_x", "left_wrist_y", "left_wrist_z"],
                                   ["left_middle_metacarpal_ex", "left_middle_metacarpal_ey", "left_middle_metacarpal_ez"])
            l_hand_sizes_wrist_mid.extend(l_wrist_mid)
            l_palm_w = get_dist(df.loc[l_val_idx],
                                ["left_index_metacarpal_ex", "left_index_metacarpal_ey", "left_index_metacarpal_ez"],
                                ["left_pinky_metacarpal_ex", "left_pinky_metacarpal_ey", "left_pinky_metacarpal_ez"])
            l_hand_sizes_palm_width.extend(l_palm_w)
            l_mid_bone = get_dist(df.loc[l_val_idx],
                                 ["left_middle_metacarpal_sx", "left_middle_metacarpal_sy", "left_middle_metacarpal_sz"],
                                 ["left_middle_metacarpal_ex", "left_middle_metacarpal_ey", "left_middle_metacarpal_ez"])
            l_hand_sizes_mid_bone.extend(l_mid_bone)
            
            ldx, ldy, ldz = df.loc[l_val_idx, "left_palm_dx"], df.loc[l_val_idx, "left_palm_dy"], df.loc[l_val_idx, "left_palm_dz"]
            lnx, lny, lnz = df.loc[l_val_idx, "left_palm_nx"], df.loc[l_val_idx, "left_palm_ny"], df.loc[l_val_idx, "left_palm_nz"]
            l_pitch = np.arctan2(-ldy, np.sqrt(ldx**2 + ldz**2))
            l_yaw = np.arctan2(ldx, ldz)
            l_roll = np.arctan2(lnx, -lny)
            l_orientations.append(np.column_stack([np.degrees(l_roll), np.degrees(l_pitch), np.degrees(l_yaw)]))
            
        # Right Hand
        r_val_idx = df[right_visible].index
        if len(r_val_idx) > 0:
            r_wrist_mid = get_dist(df.loc[r_val_idx], 
                                   ["right_wrist_x", "right_wrist_y", "right_wrist_z"],
                                   ["right_middle_metacarpal_ex", "right_middle_metacarpal_ey", "right_middle_metacarpal_ez"])
            r_hand_sizes_wrist_mid.extend(r_wrist_mid)
            r_palm_w = get_dist(df.loc[r_val_idx],
                                ["right_index_metacarpal_ex", "right_index_metacarpal_ey", "right_index_metacarpal_ez"],
                                ["right_pinky_metacarpal_ex", "right_pinky_metacarpal_ey", "right_pinky_metacarpal_ez"])
            r_hand_sizes_palm_width.extend(r_palm_w)
            r_mid_bone = get_dist(df.loc[r_val_idx],
                                 ["right_middle_metacarpal_sx", "right_middle_metacarpal_sy", "right_middle_metacarpal_sz"],
                                 ["right_middle_metacarpal_ex", "right_middle_metacarpal_ey", "right_middle_metacarpal_ez"])
            r_hand_sizes_mid_bone.extend(r_mid_bone)
            
            rdx, rdy, rdz = df.loc[r_val_idx, "right_palm_dx"], df.loc[r_val_idx, "right_palm_dy"], df.loc[r_val_idx, "right_palm_dz"]
            rnx, rny, rnz = df.loc[r_val_idx, "right_palm_nx"], df.loc[r_val_idx, "right_palm_ny"], df.loc[r_val_idx, "right_palm_nz"]
            r_pitch = np.arctan2(-rdy, np.sqrt(rdx**2 + rdz**2))
            r_yaw = np.arctan2(rdx, rdz)
            r_roll = np.arctan2(rnx, -rny)
            r_orientations.append(np.column_stack([np.degrees(r_roll), np.degrees(r_pitch), np.degrees(r_yaw)]))
            
        hand_size_records.append({
            "User": USER_MAP[user],
            "Sequence": seq_name,
            "Left_WristMid": np.mean(l_hand_sizes_wrist_mid) if l_hand_sizes_wrist_mid else np.nan,
            "Left_PalmWidth": np.mean(l_hand_sizes_palm_width) if l_hand_sizes_palm_width else np.nan,
            "Left_MiddleBone": np.mean(l_hand_sizes_mid_bone) if l_hand_sizes_mid_bone else np.nan,
            "Right_WristMid": np.mean(r_hand_sizes_wrist_mid) if r_hand_sizes_wrist_mid else np.nan,
            "Right_PalmWidth": np.mean(r_hand_sizes_palm_width) if r_hand_sizes_palm_width else np.nan,
            "Right_MiddleBone": np.mean(r_hand_sizes_mid_bone) if r_hand_sizes_mid_bone else np.nan,
        })
        
        l_ori_mean = np.mean(np.vstack(l_orientations), axis=0) if l_orientations else [np.nan, np.nan, np.nan]
        r_ori_mean = np.mean(np.vstack(r_orientations), axis=0) if r_orientations else [np.nan, np.nan, np.nan]
        orientation_records.append({
            "User": USER_MAP[user],
            "Sequence": seq_name,
            "Left_Roll": l_ori_mean[0], "Left_Pitch": l_ori_mean[1], "Left_Yaw": l_ori_mean[2],
            "Right_Roll": r_ori_mean[0], "Right_Pitch": r_ori_mean[1], "Right_Yaw": r_ori_mean[2],
        })
        
        # 3. Feature Distribution Analysis
        raw_features = df.reindex(columns=FEATURE_KEYS).fillna(0.0).to_numpy(dtype=np.float32)
        norm_features = palm_reference_normalize_sequence(raw_features)
        
        for hand in HANDS:
            is_visible = left_visible if hand == "left" else right_visible
            val_indices = np.where(is_visible)[0]
            if len(val_indices) == 0:
                continue
            if len(val_indices) > 150:
                sampled_idx = np.random.choice(val_indices, size=150, replace=False)
            else:
                sampled_idx = val_indices
                
            for idx in sampled_idx:
                frame_data = norm_features[idx]
                feat_sample = {}
                for joint_name, (jx, jy, jz) in representative_joints.items():
                    feat_sample[f"{hand}_{joint_name}_x"] = frame_data[FEATURE_INDEX[f"{hand}_{jx}"]]
                    feat_sample[f"{hand}_{joint_name}_y"] = frame_data[FEATURE_INDEX[f"{hand}_{jy}"]]
                    feat_sample[f"{hand}_{joint_name}_z"] = frame_data[FEATURE_INDEX[f"{hand}_{jz}"]]
                feat_sample["Hand"] = hand
                feature_dist_samples[user].append(feat_sample)

df_hand_size = pd.DataFrame(hand_size_records)
df_orientation = pd.DataFrame(orientation_records)
df_missing = pd.DataFrame(missing_frame_records)
df_features = pd.concat([pd.DataFrame(feature_dist_samples[u]).assign(User=USER_MAP[u]) for u in USERS], ignore_index=True)

## 1. Hand Size Analysis
We check if User 1 exhibits significantly different hand dimensions compared to the other users.

In [ ]:
for metric in ["Left_WristMid", "Left_PalmWidth", "Right_WristMid", "Right_PalmWidth"]:
    print(f"\n--- {metric} (mm) ---")
    stats = df_hand_size.groupby("User")[metric].agg(["mean", "std", "min", "max", "count"])
    print(stats.to_string())

# Boxplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
metrics = ["Left_WristMid", "Left_PalmWidth", "Right_WristMid", "Right_PalmWidth"]
titles = ["Left Hand Wrist->MCP", "Left Hand Palm Width", "Right Hand Wrist->MCP", "Right Hand Palm Width"]

for i, m in enumerate(metrics):
    sns.boxplot(data=df_hand_size, x="User", y=m, ax=axes[i], palette=COLORS, showmeans=True,
                meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":8})
    axes[i].set_title(titles[i])
    axes[i].set_ylabel("Distance (mm)")
    axes[i].set_xlabel("User")

plt.suptitle("Comparison of Hand Size Dimensions across Users")
plt.tight_layout()
plt.show()

## 2. Palm Orientation Analysis
We investigate the roll, pitch, and yaw distributions, paying attention to the Left Hand Yaw.

In [ ]:
for angle in ["Left_Roll", "Left_Pitch", "Left_Yaw", "Right_Roll", "Right_Pitch", "Right_Yaw"]:
    print(f"\n--- {angle} (degrees) ---")
    stats = df_orientation.groupby("User")[angle].agg(["mean", "std", "min", "max"])
    print(stats.to_string())

# Shift Left Yaw to [-90, 270] to resolve wrapping near 180 degrees
df_orientation["Left_Yaw_Shifted"] = np.mod(df_orientation["Left_Yaw"] + 90, 360) - 90
print("\n--- Left Hand Yaw Shifted [-90, 270] ---")
print(df_orientation.groupby("User")["Left_Yaw_Shifted"].agg(["mean", "std", "min", "max"]).to_string())

# Plot Left and Right Hand Yaw
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df_orientation, x="User", y="Left_Yaw_Shifted", ax=axes[0], palette=COLORS)
axes[0].set_title("Left Hand Yaw (Shifted to [-90, 270])")
axes[0].set_ylabel("Angle (degrees)")

sns.boxplot(data=df_orientation, x="User", y="Right_Yaw", ax=axes[1], palette=COLORS)
axes[1].set_title("Right Hand Yaw")
axes[1].set_ylabel("Angle (degrees)")

plt.suptitle("Palm Yaw Orientation comparison (Left vs Right Hand)")
plt.tight_layout()
plt.show()

## 3. Feature Distribution Analysis
We analyze the normalized features (relative to the palm origin) which are directly input to the Transformer model.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))

# KDE plots for Right Index Tip and Right Wrist (x, y, z axes)
for i, axis in enumerate(["x", "y", "z"]):
    col_tip = f"right_Index Tip_{axis}"
    col_wrist = f"right_Wrist_{axis}"
    
    # Index Tip KDE
    for idx, u in enumerate(USERS):
        user_lbl = USER_MAP[u]
        sns.kdeplot(df_features[df_features["User"] == user_lbl][col_tip].dropna(), 
                    ax=axes[i, 0], label=user_lbl, color=COLORS[idx], fill=True, alpha=0.15)
    axes[i, 0].set_title(f"Right Index Tip Normalized {axis.upper()}")
    axes[i, 0].set_xlabel("Coordinate")
    axes[i, 0].legend(fontsize=10)
    
    # Wrist KDE
    for idx, u in enumerate(USERS):
        user_lbl = USER_MAP[u]
        sns.kdeplot(df_features[df_features["User"] == user_lbl][col_wrist].dropna(), 
                    ax=axes[i, 1], label=user_lbl, color=COLORS[idx], fill=True, alpha=0.15)
    axes[i, 1].set_title(f"Right Wrist Normalized {axis.upper()}")
    axes[i, 1].set_xlabel("Coordinate")
    axes[i, 1].legend(fontsize=10)

plt.suptitle("Normalized Feature Distributions (KDE plots) for Right Hand Joints")
plt.tight_layout()
plt.show()

## 4. Missing Frame Analysis
We investigate whether User 1 has significantly more tracking dropouts than other users.

In [ ]:
print("\n--- Missing Frame Summary stats ---")
missing_stats = df_missing.groupby("User").agg({
    "TotalFrames": "sum",
    "MissingFrames": ["sum", "mean", "max"],
    "PctMissing": ["mean", "max", "std"],
    "LongestConsecutiveMissing": ["mean", "max"]
})
print(missing_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df_missing, x="User", y="PctMissing", ax=axes[0], palette=COLORS, showmeans=True)
axes[0].set_title("Percentage of Missing Frames per Sequence")
axes[0].set_ylabel("Percentage (%)")

sns.boxplot(data=df_missing, x="User", y="LongestConsecutiveMissing", ax=axes[1], palette=COLORS, showmeans=True)
axes[1].set_title("Longest Consecutive Missing Frame Segment")
axes[1].set_ylabel("Frames")

plt.suptitle("Missing Frame Analysis across Users")
plt.tight_layout()
plt.show()

## Summary of Findings

1. **Hand Size Outlier**: **User 1 has significantly larger hand size dimensions** than the other users (Left Hand Wrist→Middle MCP distance = **95.71 mm** vs **85.72 - 91.33 mm** for others). Because palm-reference normalization only subtracts the palm position and does not scale by bone length, this results in joint coordinate features that extend significantly further from the palm center (e.g., Index Tip Z extends down to **-85.73 mm** vs **-73.9 to -80.3 mm** for others), creating a covariate shift.

2. **Severe Sensor Dropouts**: **User 1 has a massive amount of completely missing frames** (332 total frames, up to **12.0%** missing in a single sequence, with a consecutive dropout segment of up to **27 frames**) due to Leap Motion tracking loss. Other users have virtually no missing frames (max 13 total, max 0.6% in a sequence, max 4 consecutive frames). A gap of 27 consecutive frames (almost 1 second of sign performance) filled with 0.0 values introduces severe discontinuities that break the Transformer sequence modeling.

3. **Palm Orientation Instability**: Even when correcting for wrapping artifacts, **User 1's Left Hand Yaw standard deviation is extremely high** (**86.99 degrees** vs **11.02 - 27.02 degrees** for other users). This indicates substantial orientation instability and tracking noise during User 1's recording session.